# Dependancies

In [ ]:
!pip install kagglehub numpy pandas matplotlib seaborn scikit-learn imbalanced-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from torch.utils.data import TensorDataset, DataLoader

NUM_FOLDS = 4
RANDOM_STATE = 42
SAMPLE_SIZE = 10000 # Edit for final run

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #used to automatically detect GPU or default to CPU otherwise

### Metric Helpers
These are just some metric helpers that we can use of need be

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
  return{
      "precision": precision_score(y_true, y_pred),
      "recall": recall_score(y_true, y_pred),
      "f1": f1_score(y_true, y_pred),
      "roc_auc": roc_auc_score(y_true, y_prob),
      "mcc": matthews_corrcoef(y_true, y_pred),
  }

def save_fig(fig, name):
    path = os.path.join(RESULTS, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def confusion_matrices(splits, Xfeatures, y, train_predict_fn, model_name, file):
    fig, axes = plt.subplots(1, len(splits), figsize=(5 * len(splits), 4))
    if len(splits) == 1:
        axes = [axes]

    for i, (train_idx, test_idx) in enumerate(splits):
        y_pred = train_predict_fn(train_idx, test_idx)
        y_test = y.loc[test_idx]
        cm = confusion_matrix(y_test, y_pred)

        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                    xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
        axes[i].set_title(f"Split {i + 1}")
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("Actual")

    fig.suptitle(f"{model_name} - Confusion Matrices", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, file)


def plot_roc_curve(splits, Xfeatures, y, train_predict_proba_fn, model_name, file):
    fig, ax = plt.subplots(figsize=(8, 6))

    for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
        y_prob = train_predict_proba_fn(train_idx, test_idx)
        y_test = y.loc[test_idx]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"Split {split_num} (AUC = {auc_val:.3f})")

    ax.plot([0, 1], [0, 1], "k--", label="Random Classifier")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} ROC Curve")
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_fig(fig, file)


def plot_metrics_bar(results_df, splits, model_name, file):
    fig, ax = plt.subplots(figsize=(10, 5))

    metrics_to_plot = ["precision", "recall", "f1", "roc_auc", "mcc"]
    x = np.arange(len(splits))
    bar_width = 0.15
    colors = ["#056517", "#de1a24", "#3b75e9", "#cad5ed", "#2832c4"]

    for i, metric in enumerate(metrics_to_plot):
        vals = results_df[metric].values
        ax.bar(x + i * bar_width, vals, bar_width, label=metric, color=colors[i])

    ax.set_xlabel("Split", fontsize=12, fontweight="bold")
    ax.set_ylabel("Score", fontsize=12, fontweight="bold")
    ax.set_title(f"{model_name} Metrics", fontsize=14, fontweight="bold")
    ax.set_xticks(x + bar_width * (len(metrics_to_plot) - 1) / 2)
    ax.set_xticklabels([f"Split {i + 1}" for i in range(len(splits))])
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_fig(fig, file)


def plot_feature_coefficients(coefs, feature_names, model_name, split_label, file, top_n=15):
    fig, ax = plt.subplots(figsize=(12, 6))

    feature_names = np.array(feature_names)
    sorted_idx = np.argsort(np.abs(coefs))[::-1]
    top_idx = sorted_idx[:top_n]

    top_coefs = coefs[top_idx]
    top_names = feature_names[top_idx]
    bar_colors = ["#de1a24" if c < 0 else "#056517" for c in top_coefs]

    ax.barh(top_names, top_coefs, color=bar_colors, edgecolor="black")
    ax.set_xlabel("Coefficient Value", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Coefficients ({split_label})",
                 fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, file)

def _make_cached_predictor(fold_predictions, key, n_folds):
    preds = [fold_predictions[i + 1][key] for i in range(n_folds)]
    call_idx = [0]
    def predictor(train_idx, test_idx):
        result = preds[call_idx[0]]
        call_idx[0] += 1
        return result
    return predictor

def create_kfold_splits(X, y, n_folds=NUM_FOLDS, random_state=RANDOM_STATE):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    return list(kf.split(X, y))

def print_classification_report(y_true, y_pred):
    print(classification_report(y_true, y_pred,target_names=["Legitimate", "Fraudulent"],zero_division=0))

def print_summary_table(results_dset2_df, model_name):
    print(f"\n {model_name} Summary Across All Folds")
    print(f"  {'Metric':<12} {'Mean':>10} {'Std':>10}")
    print(f"  {'-' * 32}")
    for metric in ["precision", "recall", "f1", "mcc", "auc_roc"]:
        mean_val = results_dset2_df[metric].mean()
        std_val = results_dset2_df[metric].std()
        print(f"  {metric:<12} {mean_val:>10.4f} {std_val:>10.4f}")

def _build_lr():
    return LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    )

def _build_rf():
    return RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs= -1
    )

# Feed-Forward Neural Network

In [ ]:
HIDDEN1 = 64
HIDDEN2 = 32
DROPOUT = 0.3
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

class FraudDetectorNN(nn.Module):
    def __init__(self, input_dim, hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)

def _set_seeds():
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)

def _train_nn(X_train, y_train, input_dim, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    _set_seeds()
    model = FraudDetectorNN(input_dim).to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=lr, weight_decay=weight_decay)
    X_arr = X_train.values if hasattr(X_train, "values") else np.array(X_train)
    y_arr = y_train.values if hasattr(y_train, "values") else np.array(y_train)
    X_tensor = torch.FloatTensor(X_arr)
    y_tensor = torch.FloatTensor(y_arr)
    dataset = TensorDataset(X_tensor, y_tensor)
    g = torch.Generator().manual_seed(RANDOM_STATE)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=g)
    epoch_losses = []
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        n_samples = 0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            y_hat = model(X_batch)
            loss = criterion(y_hat, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(X_batch)
            n_samples += len(X_batch)

        avg_loss = running_loss / n_samples
        epoch_losses.append(avg_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"      Epoch {epoch + 1:>3}/{epochs}  |  Loss: {avg_loss:.6f}")

    return model, epoch_losses

def _predict(model, X_test):
    model.eval()
    with torch.no_grad():
        X_arr = X_test.values if hasattr(X_test, "values") else np.array(X_test)
        X_tensor = torch.FloatTensor(X_arr).to(DEVICE)
        y_prob = model(X_tensor).cpu().numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    return y_pred, y_prob

def make_predictor(key):
  predictions = [split_predictions[i+1][key] for i in range(len(splits))]
  call_idx = [0]

  def predictor(train_idx, test_idx):
    result = predictions[call_idx[0]]
    call_idx[0] += 1
    return result
  return predictor

## Load All Data

In [ ]:
Dataset1 =kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "rupakroy/online-payments-fraud-detection-dataset", "PS_20174392719_1491204439457_log.csv",)

/tmp/ipykernel_7746/1713948650.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  Dataset1 =kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "rupakroy/online-payments-fraud-detection-dataset", "PS_20174392719_1491204439457_log.csv",)


Using Colab cache for faster access to the 'online-payments-fraud-detection-dataset' dataset.


In [ ]:
Dataset2 = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "nelgiriyewithana/credit-card-fraud-detection-dataset-2023","creditcard_2023.csv",)

/tmp/ipykernel_7746/2640126912.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  Dataset2 = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "nelgiriyewithana/credit-card-fraud-detection-dataset-2023","creditcard_2023.csv",)


Using Colab cache for faster access to the 'credit-card-fraud-detection-dataset-2023' dataset.


KeyboardInterrupt: 

In [ ]:
Dataset3 = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "whenamancodes/fraud-detection", "creditcard.csv",)

# Dataset 1

In [ ]:
df1 = Dataset1.sample(n=min(SAMPLE_SIZE, len(Dataset1)), random_state=42)

df1.head()

## Preprocessing

In [ ]:
RESULTS = "Dataset1 results"
os.makedirs(RESULTS, exist_ok=True)

#Cleaning dataset
missing = df1.isnull().sum()
total_missing = missing.sum()
print(f"\nMissing values: {total_missing}")
if total_missing > 0:
    print(missing[missing > 0])

# Check for duplicate rows
duplicates = df1.duplicated().sum()
print(f"Duplicate rows: {duplicates:,}")
if duplicates > 0:
    print("  → Dropping duplicates")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  → New shape: {df.shape[0]:,} rows")

# There are no missing rows or duplicates, therefore null/missing/duplicate cleaning not needed

In [ ]:
# drop identifiers(nameOrig, nameDest)
df1 = df1.drop(columns=['nameOrig', 'nameDest', 'step'])

df1.head()

# encoding "type"
if 'type' in df1.columns:
    print(f"Original 'type' column values: {df1['type'].unique()}")
    print(f"Value counts:\n{df1['type'].value_counts()}")

    label_encoder = LabelEncoder()
    df1['type_encoded'] = label_encoder.fit_transform(df1['type'])
    print(f"\nLabel encoding applied:")
    for i, category in enumerate(label_encoder.classes_):
        print(f"  {category} → {i}")

    df1 = df1.drop(columns=['type'])
    print(f"\nDropped original 'type' column")
else:
    print("No 'type' column found for encoding")

df1.head()

### Data Analysis

In [ ]:
print(f"Total transactions: {len(Dataset1):,}")
class_counts = Dataset1["isFraud"].value_counts().sort_index()
class_pct = Dataset1["isFraud"].value_counts(normalize=True).sort_index() * 100
print(f"Legitimate transactions: {class_counts[0]:,} ({class_pct[0]:.4f}%)")
print(f"Fraudulent transactions: {class_counts[1]:,} ({class_pct[1]:.4f}%)")

### Feature Engineering

In [ ]:
#Feture Engineering
df1 = df1.assign(
    balance_change_orig=df1['newbalanceOrig'] - df1['oldbalanceOrg']
)

df1 = df1.assign(
    balance_change_dest=df1['newbalanceDest'] - df1['oldbalanceDest']
)

df1 = df1.assign(
    amount_to_balance_ratio=df1['amount'] / (df1['oldbalanceOrg'] + 1)
)

df1 = df1.assign(
    amount_to_dest_ratio=df1['amount'] / (df1['oldbalanceDest'] + 1)
)

df1 = df1.assign(
    abs_balance_change_orig=np.abs(df1['balance_change_orig']),
    abs_balance_change_dest=np.abs(df1.get('balance_change_dest', 0))
)

amount_mean = df1['amount'].mean()
amount_std = df1['amount'].std()

df1 = df1.assign(
    is_large_transaction=(df1['amount'] > amount_mean + 2 * amount_std).astype(int)
)

df1 = df1.assign(
    balance_mismatch_orig=(
        np.abs(df1['newbalanceOrig'] - (df1['oldbalanceOrg'] - df1['amount'])) > 0.01
    ).astype(int)
)

df1 = df1.assign(
    is_zero_balance_after=(df1['newbalanceOrig'] == 0).astype(int)
)

df1 = df1.assign(
    type_amount_interaction=df1['type_encoded'] * np.log1p(df1['amount'])
)

df1 = df1.assign(
    amount_change_interaction=df1['amount'] * df1['balance_change_orig']
)

df1 = df1.assign(
    log_amount=np.log1p(df1['amount']),
    log_oldbalance_orig=np.log1p(df1['oldbalanceOrg']),
    log_oldbalance_dest=np.log1p(df1.get('oldbalanceDest', 0))
)

avg_transaction_amount = df1['amount'].mean()
df1 = df1.assign(
    amount_to_avg_ratio=df1['amount'] / (avg_transaction_amount + 1)
)

df1.head()

## Data Split

In [ ]:
# Features and Targets
X = df1.drop(columns=['isFraud'])
Y = df1['isFraud']

Y = Y.astype(int)

feature_names = X.columns.tolist()

print(f"Features shape: {X.shape}")
print(f"Target shape: {Y.shape}")
print(f"\nTarget distribution:")
print(f"  Non-Fraud (0): {(Y == 0).sum():,} ({(Y == 0).mean() * 100:.4f}%)")
print(f"  Fraud (1):     {(Y == 1).sum():,} ({(Y == 1).mean() * 100:.4f}%)")

def scale_resample(X, y, train_idx, test_idx):
    feature_names = X.columns.tolist()
    X_train_raw = X.iloc[train_idx]
    X_test_raw = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw))
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw))

    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

    return X_train_resampled, y_train_resampled, X_test_scaled, y_test

splits = create_kfold_splits(X, Y)

for i, (train_idx, test_idx) in enumerate(splits, start=1):
    train_fraud = Y.iloc[train_idx].sum()
    test_fraud = Y.iloc[test_idx].sum()
    test_pct = test_fraud / len(test_idx) * 100

## Models

### Logistic Regreession

In [ ]:
lr_results_ds1 = []
lr_last_model = None

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    # Get preprocessed data for this fold
    X_train_rs, y_train_rs, X_test_scaled, y_test = scale_resample(X, Y, train_idx, test_idx)

    # Train the model
    model = _build_lr()
    model.fit(X_train_rs, y_train_rs)
    lr_last_model = model  # Keep the last trained model

    # Predict and evaluate
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    lr_results_ds1.append(metrics)

# Convert results to a DataFrame for easy analysis
lr_results_ds1_df = pd.DataFrame(lr_results_ds1)
lr_results_ds1_df.to_csv(os.path.join(RESULTS, "lr_results_ds1.csv"), index=False)
print_summary_table(lr_results_ds1_df, "Logistic Regression")

In [ ]:
def _lr_predict(train_idx, test_idx):
    """Predict using Logistic Regression for a given fold"""
    X_train_rs, y_train_rs, X_test_scaled, y_test = scale_resample(X, Y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_train_rs, y_train_rs)
    return model.predict(X_test_scaled)

def _lr_predict_proba(train_idx, test_idx):
    """Predict probabilities using Logistic Regression for a given fold"""
    X_train_rs, y_train_rs, X_test_scaled, y_test = scale_resample(X, Y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_train_rs, y_train_rs)
    return model.predict_proba(X_test_scaled)[:, 1]

plot_confusion_matrices(splits, X, Y, _lr_predict,"Logistic Regression", "d1_lr_confusion_matrices.png")
plot_roc_curves(splits, X, Y, _lr_predict_proba,"Logistic Regression", "d1_lr_roc_curves.png")
plot_metrics_bars(lr_results_ds1_df, splits, "Logistic Regression","d1_lr_metrics_by_fold.png")
plot_feature_coefficients(lr_last_model.coef_[0], feature_names,"Logistic Regression", f"Fold {len(splits)}","14_lr_feature_coefficients.png")

### Random Forest Classifier

In [ ]:
#training random forest model
rf_results = []
rf_last_model = None
rf_fold_predictions = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test = scale_resample(X, Y, train_idx, test_idx)
    model = _build_rf()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    rf_results.append(metrics)
    rf_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    rf_last_model = model

rf_results_ds1_df = pd.DataFrame(rf_results)
rf_results_ds1_df.to_csv(os.path.join(RESULTS, "rf_results.csv"), index=False)
print_summary_table(rf_results_ds1_df, "Random Forest")

In [ ]:
plot_confusion_matrices(splits, X, Y,_make_cached_predictor(rf_fold_predictions, "y_pred", len(splits)),"Random Forest", "d1_rf_confusion_matrices.png")
plot_roc_curves(splits, X, Y,_make_cached_predictor(rf_fold_predictions, "y_prob", len(splits)),"Random Forest", "d1_rf_roc_curves.png")
plot_metrics_bars(rf_results_ds1_df, splits, "Random Forest", "d1_rf_metrics_by_fold.png")
plot_feature_importances(rf_last_model.feature_importances_, feature_names,"Random Forest", f"Fold {len(splits)}","d1_rf_feature_importances.png")

### Feed-Forward Neural Network

In [ ]:
input_dim = X.shape[1]
nn_results = []
nn_fold_predictions = {}
all_epoch_losses = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test = scale_resample(X, Y, train_idx, test_idx)
    model, epoch_losses = _train_nn(X_train, y_train, input_dim)
    y_pred, y_prob = _predict(model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    nn_results.append(metrics)
    nn_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    all_epoch_losses[split_num] = epoch_losses

nn_results_ds1_df = pd.DataFrame(nn_results)
nn_results_ds1_df.to_csv(os.path.join(RESULTS, "nn_results.csv"), index=False)
print_summary_table(nn_results_ds1_df, "Feed Forward Neural Network")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("tab10")
for split_num in sorted(all_epoch_losses.keys()):
    losses = all_epoch_losses[split_num]
    ax.plot(range(1, len(losses) + 1), losses,
            label=f"Fold {split_num}", color=cmap((split_num - 1) % 10), linewidth=1.3)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
ax.set_title("Feed-Forward Neural Network - Training Loss Curves",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8); ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "d1_nn_training_loss.png")

plot_confusion_matrices(splits, X, Y,_make_cached_predictor(nn_fold_predictions, "y_pred", len(splits)),"Feed-Forward Neural Network", "d1_nn_confusion_matrices.png")
plot_roc_curves(splits, X, Y,_make_cached_predictor(nn_fold_predictions, "y_prob", len(splits)),"Feed-Forward Neural Network", "d1_nn_roc_curves.png")
plot_metrics_bars(nn_results_ds1_df, splits, "Feed-Forward Neural Network","d1_nn_metrics_by_fold.png")
print("\nFeed-Forward Neural Network complete.")

In [ ]:
summary_frames = {"Logistic Regression": lr_results_ds1_df,"Random Forest": rf_results_ds1_df,"Feed-Forward NN": nn_results_ds1_df}
metric_cols = ["precision", "recall", "f1", "mcc", "roc_auc"]
summary = pd.DataFrame({
    name: frame[metric_cols].mean()
    for name, frame in summary_frames.items()
}).T

print("Mean metrics across all 4 folds:")
print(summary.round(4))
summary.to_csv(os.path.join(RESULTS, "summary_mean_metrics.csv"))

#Dataset 2

##Preprocessing

In [ ]:
dset2_df = Dataset2.sample(n=min(SAMPLE_SIZE, len(Dataset2)), random_state=42) #Edit for final run
dset2_df.head()

In [ ]:
RESULTS = "Dataset2 results"
os.makedirs(RESULTS, exist_ok=True)

###Dataset Cleaning

In [ ]:
# drop ID column to prevent memorization
if "id" in dset2_df.columns:
    dset2_df = dset2_df.drop(columns=["id"])

# check for missing values
missing = dset2_df.isnull().sum()
total_missing = missing.sum()
print(f"\nMissing values: {total_missing}")
if total_missing > 0:
    print(missing[missing > 0])

# check for duplicate rows to prevent information leakage
duplicates = dset2_df.duplicated().sum()
print(f"Duplicate rows: {duplicates:,}")
if duplicates > 0:
    dset2_df = dset2_df.drop_duplicates().reset_index(drop=True)
    print(f"  -> New shape: {dset2_df.shape[0]:,} rows")

###Dataset Analysis

In [ ]:
class_counts = dset2_df["Class"].value_counts().sort_index()
class_pct = dset2_df["Class"].value_counts(normalize=True).sort_index() * 100
colors = ["#7ed957", "#d236b4"]
labels = ["Legitimate (0)", "Fraudulent (1)"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(labels, class_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Transaction Class Counts", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pct.values)):
    axes[0].text(i, cnt + cnt * 0.01, f"{cnt:,}\n({pct:.3f}%)", ha="center", fontsize=10)

axes[1].pie(class_counts.values, labels=labels, colors=colors, autopct="%1.3f%%", startangle=90)
axes[1].set_title("Class Proportion", fontsize=14, fontweight="bold")

fig.suptitle("Pre-Balanced Class Distribution in Dataset 2 (~50/50)",fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "Dataset2_class_distribution.png")

In [ ]:
legit = dset2_df[dset2_df["Class"] == 0]["Amount"]
fraud = dset2_df[dset2_df["Class"] == 1]["Amount"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(legit, bins=80, alpha=0.6, color="#7ed957", label="Legitimate", density=True)
axes[0].hist(fraud, bins=80, alpha=0.6, color="#db2a9d", label="Fraudulent", density=True)
axes[0].set_title("Amount Distribution (Full Range)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Amount"); axes[0].set_ylabel("Density"); axes[0].legend()

#given that the majority of transactions seem to appear below $500 this range is used for cleaerer visualization
axes[1].hist(legit[legit <= 500], bins=80, alpha=0.6, color="#7ed957", label="Legitimate", density=True)
axes[1].hist(fraud[fraud <= 500], bins=80, alpha=0.6, color="#d236b4", label="Fraudulent", density=True)
axes[1].set_title("Amount Distribution (≤ $500)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Amount"); axes[1].set_ylabel("Density"); axes[1].legend()

fig.suptitle("Transaction Amount: Legitimate vs Fraudulent", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "Dataset2_amount_distribution.png")


In [ ]:
v_cols = [f"V{i}" for i in range(1, 29)]
feature_cols = v_cols + ["Amount"]
corr_with_class = (
    dset2_df[feature_cols + ["Class"]].corr()["Class"].drop("Class").sort_values()
)

print("  Features most negatively correlated with Class(fraud):")
for feat, corr in corr_with_class.head(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

print("\n  Features most positively correlated with Class(fraud):")
for feat, corr in corr_with_class.tail(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

#the green bar increases a prediction of fraud while the red increses the prediction of a legitimate transaction
fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ["#d236b4" if v < 0 else "#7ed957" for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Pearson Correlation with Class", fontsize=12)
ax.set_title("Feature Correlation with Fraud Label", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.8)
fig.tight_layout()
save_fig(fig, "Dataset2_feature_correlation_with_class.png")


###Preparing Data for Training Models

In [ ]:
X = dset2_df.drop(columns=["Class"])
y = dset2_df["Class"]

In [ ]:
#dataset already balanced hence no SMOTE
def scale_split(X, y, train_idx, test_idx):
    feature_names = X.columns.tolist()
    X_train_raw = X.iloc[train_idx]
    X_test_raw = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_raw),
        columns=feature_names, index=X_train_raw.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test_raw),
        columns=feature_names, index=X_test_raw.index,
    )

    return X_train_scaled, y_train, X_test_scaled, y_test, scaler

splits = create_kfold_splits(X, y)
for i, (train_idx, test_idx) in enumerate(splits, start=1):
    train_fraud = y.iloc[train_idx].sum()
    test_fraud = y.iloc[test_idx].sum()
    test_pct = test_fraud / len(test_idx) * 100

###Evaluation Helpers

In [ ]:
def plot_confusion_matrices(splits, X_features, y, train_and_predict_fn, model_name, filename):
    fig, axes = plt.subplots(1, len(splits), figsize=(4 * len(splits), 4))
    if len(splits) == 1:
        axes = [axes]

    for i, (train_idx, test_idx) in enumerate(splits, start=1):
        y_pred = train_and_predict_fn(train_idx, test_idx)
        y_test = y.iloc[test_idx]
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i - 1], xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"], cbar=False)
        axes[i - 1].set_title(f"Fold {i}", fontsize=11, fontweight="bold")
        axes[i - 1].set_ylabel("Actual"); axes[i - 1].set_xlabel("Predicted")

    fig.suptitle(f"{model_name} - Confusion Matrices", fontsize=16, fontweight="bold", y=1.02)
    fig.tight_layout()
    save_fig(fig, filename)

def plot_roc_curves(splits, X_features, y, train_and_predict_proba_fn, model_name, filename):
    fig, ax = plt.subplots(figsize=(8, 6))

    for i, (train_idx, test_idx) in enumerate(splits, start=1):
        y_prob = train_and_predict_proba_fn(train_idx, test_idx)
        y_test = y.iloc[test_idx]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"Fold {i} (AUC={auc_val:.4f})", alpha=0.7)

    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Classifier")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"{model_name} ROC Curves", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    save_fig(fig, filename)

def plot_metrics_bars(results_dset2_df, splits, model_name, filename):
    fig, ax = plt.subplots(figsize=(14, 5))
    metrics_to_plot = ["precision", "recall", "f1", "mcc", "roc_auc"]
    x = np.arange(len(splits))
    width = 0.15
    colors_metrics = ["#3498db", "#7ed957", "#d236b4", "#9b59b6", "#f39c12"]

    for i, metric in enumerate(metrics_to_plot):
        ax.bar(x + i * width, results_dset2_df[metric].values, width,
               label=metric.upper().replace("_", "-"), color=colors_metrics[i])

    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"{model_name} - Metrics by Fold", fontsize=14, fontweight="bold")
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels([f"F{i+1}" for i in range(len(splits))])
    ax.legend(loc="lower right"); ax.set_ylim(0, 1.05)
    fig.tight_layout()
    save_fig(fig, filename)

def plot_feature_coefficients(coefs, feature_names, model_name, fold_label, filename, top_n=15):
    fig, ax = plt.subplots(figsize=(12, 6))
    sorted_idx = np.argsort(np.abs(coefs))[::-1]
    ax.barh(range(top_n), coefs[sorted_idx[:top_n]], color=["#d236b4" if c < 0 else "#7ed957"
                   for c in coefs[sorted_idx[:top_n]]],edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx[:top_n]])
    ax.set_xlabel("Coefficient Value", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Feature Coefficients ({fold_label})", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, filename)

def plot_feature_importances(importances, feature_names, model_name, fold_label, filename, top_n=15):
    fig, ax = plt.subplots(figsize=(12, 6))
    sorted_idx = np.argsort(importances)[::-1][:top_n]
    ax.barh(range(top_n), importances[sorted_idx], color="#3498db", edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx])
    ax.set_xlabel("Gini Importance", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Feature Importances ({fold_label})", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, filename)

##Logistic Regression

In [ ]:
feature_names = X.columns.tolist()
lr_results = []
lr_last_model = None

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    lr_results.append(metrics)
    lr_last_model = model

lr_dset2_df = pd.DataFrame(lr_results)
lr_dset2_df.to_csv(os.path.join(RESULTS, "lr_results.csv"), index=False)
print_summary_table(lr_dset2_df, "Logistic Regression")

In [ ]:
def _lr_predict(train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_split(X, y, train_idx, test_idx)
    m = _build_lr(); m.fit(X_tr, y_tr)
    return m.predict(X_te)

def _lr_predict_proba(train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_split(X, y, train_idx, test_idx)
    m = _build_lr(); m.fit(X_tr, y_tr)
    return m.predict_proba(X_te)[:, 1]

plot_confusion_matrices(splits, X, y, _lr_predict,"Logistic Regression", "11_lr_confusion_matrices.png")
plot_roc_curves(splits, X, y, _lr_predict_proba,"Logistic Regression", "12_lr_roc_curves.png")
plot_metrics_bars(lr_dset2_df, splits, "Logistic Regression","13_lr_metrics_by_fold.png")
plot_feature_coefficients(lr_last_model.coef_[0], feature_names,"Logistic Regression", f"Fold {len(splits)}","14_lr_feature_coefficients.png")

##Random forest

In [ ]:
rf_results = []
rf_last_model = None
rf_fold_predictions = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    model = _build_rf()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    rf_results.append(metrics)
    rf_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    rf_last_model = model

rf_dset2_df = pd.DataFrame(rf_results)
rf_dset2_df.to_csv(os.path.join(RESULTS, "rf_results.csv"), index=False)
print_summary_table(rf_dset2_df, "Random Forest")


In [ ]:
plot_confusion_matrices(splits, X, y,_make_cached_predictor(rf_fold_predictions, "y_pred", len(splits)),"Random Forest", "15_rf_confusion_matrices.png")
plot_roc_curves(splits, X, y,_make_cached_predictor(rf_fold_predictions, "y_prob", len(splits)),"Random Forest", "16_rf_roc_curves.png")
plot_metrics_bars(rf_dset2_df, splits, "Random Forest", "17_rf_metrics_by_fold.png")
plot_feature_importances(rf_last_model.feature_importances_, feature_names,"Random Forest", f"Fold {len(splits)}","18_rf_feature_importances.png")

##Feed-Forward Neural Network

In [ ]:
input_dim = X.shape[1]
nn_results = []
nn_fold_predictions = {}
all_epoch_losses = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    model, epoch_losses = _train_nn(X_train, y_train, input_dim)
    y_pred, y_prob = _predict(model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    nn_results.append(metrics)
    nn_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    all_epoch_losses[split_num] = epoch_losses

nn_dset2_df = pd.DataFrame(nn_results)
nn_dset2_df.to_csv(os.path.join(RESULTS, "nn_results.csv"), index=False)
print_summary_table(nn_dset2_df, "Feed-Forward Neural Network")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("tab10")
for split_num in sorted(all_epoch_losses.keys()):
    losses = all_epoch_losses[split_num]
    ax.plot(range(1, len(losses) + 1), losses,
            label=f"Fold {split_num}", color=cmap((split_num - 1) % 10), linewidth=1.3)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
ax.set_title("Feed-Forward Neural Network - Training Loss Curves",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8); ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "19_nn_training_loss.png")

plot_confusion_matrices(splits, X, y,_make_cached_predictor(nn_fold_predictions, "y_pred", len(splits)),"Feed-Forward Neural Network", "20_nn_confusion_matrices.png")
plot_roc_curves(splits, X, y,_make_cached_predictor(nn_fold_predictions, "y_prob", len(splits)),"Feed-Forward Neural Network", "21_nn_roc_curves.png")
plot_metrics_bars(nn_dset2_df, splits, "Feed-Forward Neural Network","22_nn_metrics_by_fold.png")
print("\nFeed-Forward Neural Network complete.")


In [ ]:
summary_frames = {"Logistic Regression": lr_dset2_df,"Random Forest": rf_dset2_df,"Feed-Forward NN": nn_dset2_df}
metric_cols = ["precision", "recall", "f1", "mcc", "roc_auc"]
summary = pd.DataFrame({
    name: frame[metric_cols].mean()
    for name, frame in summary_frames.items()
}).T

print("Mean metrics across all 4 folds:")
print(summary.round(4))
summary.to_csv(os.path.join(RESULTS, "summary_mean_metrics.csv"))

# Dataset 3

### Plan of Action
1. Load Data from kaggle
2. Clean and display information about it
3. Time-based expanding window splits and SMOTE(This is on training data only btw)
4. Logistic Regression - 4 Plots
5. Random Forest
6. FFNN

In [ ]:
RD = "sample_data/results/Dataset3"
os.makedirs(RD, exist_ok=True)

## Data Cleaning

In [ ]:
Dataset3["Class"] = Dataset3["Class"].astype(int)
missing = Dataset3.isnull().sum()
total_missing = missing.sum()
if total_missing>0:
  Dataset3 = Dataset3.dropna().reset_index(drop=True)
duplicates = Dataset3.duplicated().sum()
if duplicates>0:
  Dataset3 = Dataset3.drop_duplicates().reset_index(drop=True)


# Data Analysis

## Class Distribution in Bar and Pie Chart

In [ ]:
class_counts = Dataset3["Class"].value_counts().sort_index()
class_pct = Dataset3["Class"].value_counts(normalize=True).sort_index() * 100

colors = ["#056517","#de1a24"]
labels = ["Legitimate", "Fraudulent"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(labels, class_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Transaction Class Counts", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i in range(len(labels)):
    count = class_counts.values[i]
    pct   = class_pct.values[i]
    y_position = count + count * 0.01
    label_text = f"{count:,}\n({pct:.3f}%)"
    axes[0].text(i, y_position, label_text, ha="center", fontsize=10)

axes[1].pie(class_counts.values,labels=labels,colors=colors,autopct="%1.3f%%",startangle=90,explode=(0, 0.1),)
axes[1].set_title("Class Proportion", fontsize=14, fontweight = "bold")
save_fig(fig, "Dataset_Class_Distribution")

# All Transactions
These two graphs split up all transactions over time and all fraudulent transactions over the same length of time.

In [ ]:
Dataset3["Hour"] = Dataset3["Time"]/3600
total_hours = Dataset3["Hour"].max()
fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharex = True) # Sharex
axes[0].hist(Dataset3["Hour"], bins = 48, color = "#056517", edgecolor = "black")
axes[0].set_title(" All Transactions Over Time", fontsize = 14, fontweight = "bold")
axes[0].set_xlabel("Number of Transactions")

fraud_hours = Dataset3[Dataset3["Class"] == 1]["Hour"]
axes[1].hist(fraud_hours, bins = 48, color = "#de1a24", edgecolor = "black")
axes[1].set_title("Fraudulent Transactions Over Time", fontsize = 14, fontweight = "bold")
axes[1].set_ylabel("Number of Transactions")
axes[1].set_xlabel("Time")
save_fig(fig, "All_Transactions_Over_Time")


#Feature Correlation Chart
This cell is measuring which features have the strongest linear correlation with the fraud label and plots it out.

In [ ]:
v_cols = [f"V{i}" for i in range (1,29)]
feature_cols = v_cols + ["Amount"]
corr_class = (Dataset3[feature_cols + ["Class"]].corr()["Class"].drop("Class").sort_values())
fig, ax = plt.subplots(figsize = (12,6))
bar_colors = ["#de1a24" if v<0 else "#056517" for v in corr_class.values]
ax.barh(corr_class.index, corr_class.values, color = bar_colors, edgecolor = "black")
ax.set_xlabel("Peasron Correlation with Class")
ax.set_title("Feature Correlation with Class", fontsize = 14, fontweight = "bold")
save_fig(fig, "Feature_Correlation_Chart")

#Preprocessing and Time Splits

In [ ]:
Dataset3.drop(columns = ["Hour"], inplace=True, errors ="ignore")#inplace
X = Dataset3.drop(columns = ["Class"])
y = Dataset3["Class"]

def create_time_splits(X, test_window = 8, train_hours=16):
  hours = X["Time"]/3600
  total_hours = hours.max()
  splits = []
  test_start = train_hours

  while test_start < total_hours:
    test_end = min(test_start +test_window, total_hours+1)
    train_mask = hours<test_start
    test_mask = (hours>=test_start) & (hours<test_end)
    train_idx = X[train_mask].index.tolist()
    test_idx = X[test_mask].index.tolist()
    if len(train_idx)>0 and len(test_idx) >0:
      splits.append((train_mask, test_mask))

    test_start += test_window
  return splits


def scale_resample(Xfeatures, y, train_idx, test_idx):
  feature_names = Xfeatures.columns.tolist()

  X_trainr = Xfeatures.loc[train_idx]
  X_testr = Xfeatures.loc[test_idx]
  y_trainr = y.loc[train_idx]
  y_testr = y.loc[test_idx]
  scaler = StandardScaler()

  XtrainScaled = pd.DataFrame(scaler.fit_transform(X_trainr), columns= feature_names, index=X_trainr.index)
  XtestScaled = pd.DataFrame(scaler.transform(X_testr), columns = feature_names, index = X_testr.index)

  smote = SMOTE(random_state=42)
  X_train_resampled, y_train_resampled = smote.fit_resample(XtrainScaled, y_trainr)

  return X_train_resampled, y_train_resampled, XtestScaled, y_testr

splits = create_time_splits(X)

# MODELS

## Logistic Regression

In [ ]:
Xfeatures = X.drop(columns = ["Time"])
feature_names = Xfeatures.columns.tolist()
lr_results = []
lm = None
for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
  X_train_resampled, y_train_resampled, XtestScaled, y_testr = scale_resample(Xfeatures, y, train_idx, test_idx)
  model = _build_lr()
  model.fit(X_train_resampled, y_train_resampled)
  y_pred = model.predict(XtestScaled)
  y_prob = model.predict_proba(XtestScaled)[:,1]
  metrics = compute_metrics(y_testr, y_pred, y_prob)
  lr_results.append(metrics)
lm = model
lr_results = pd.DataFrame(lr_results)

### Visualizations

In [ ]:
def _lr_train_predict(train_idx, test_idx):
    X_tr, y_tr, X_te, _, = scale_resample(Xfeatures, y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_tr, y_tr)
    return model.predict(X_te)

def _lr_train_predict_proba(train_idx, test_idx):
    X_tr, y_tr, X_te, _, = scale_resample(Xfeatures, y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_te)[:, 1]

confusion_matrices(splits, Xfeatures, y, _lr_train_predict, "Logistic Regression", "Logistic_Regression_Confusion_Matrix")
plot_roc_curve(splits, Xfeatures, y, _lr_train_predict_proba,"Logistic Regression", "Logistic_Regression_ROC_Curve")
plot_metrics_bar(pd.DataFrame(lr_results), splits,"Logistic Regression", "Logistic_Regression_Metrics")
plot_feature_coefficients(lm.coef_[0], feature_names,"Logistic Regression", f"Split {len(splits)}", "Datadet3_LR_feature_coefficients.png")

# Random Forest

In [ ]:
rf_results = []
rf_last_model = None

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
  X_train_resampled, y_train_resampled, XtestScaled, y_testr = scale_resample(Xfeatures, y, train_idx, test_idx)
  model = _build_rf()
  model.fit(X_train_resampled, y_train_resampled)
  y_pred = model.predict(XtestScaled)
  y_prob = model.predict_proba(XtestScaled)[:, 1]
  metrics = compute_metrics(y_testr, y_pred, y_prob)
  metrics["split"] = split_num
  rf_results.append(metrics)
  rf_last_model = model

rf_df = pd.DataFrame(rf_results)

### Visualizations

In [ ]:
def _rf_train_predict(train_idx, test_idx):
  X_tr, y_tr, X_te, _ = scale_resample(Xfeatures, y, train_idx, test_idx)
  m = _build_rf()
  m.fit(X_tr, y_tr)
  return m.predict(X_te)

def _rf_train_predict_proba(train_idx, test_idx):
  X_tr, y_tr, X_te, _ = scale_resample(Xfeatures, y, train_idx, test_idx)
  m = _build_rf()
  m.fit(X_tr, y_tr)
  return m.predict_proba(X_te)[:, 1]

confusion_matrices(splits, Xfeatures, y, _rf_train_predict, "Random Forest", "RandomForestConfusionMatrices.png")
plot_roc_curve(splits, Xfeatures, y, _rf_train_predict_proba, "Random Forest", "RandomForestROC.png")
plot_metrics_bar(rf_df, splits, "Random Forest", "RandomForestMetrics.png")

importances = rf_last_model.feature_importances_
top_n = 15
sorted_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(top_n), importances[sorted_idx], color=["#de1a24" if i < 0 else "#056517" for i in importances[sorted_idx]], edgecolor="black")
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("Importance")
ax.set_title("Random Forest Feature Importances", fontsize=14, fontweight="bold")
fig.tight_layout()
save_fig(fig, "RandomForestFeatureImportances.png")

## FFNN Training, Evaluations and Visuals

In [ ]:
input_dim = Xfeatures.shape[1]
nn_results = []
split_predictions = {}
all_epoch_losses = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
  X_train_resampled, y_train_resampled, XtestScaled, y_testr = scale_resample(Xfeatures, y, train_idx, test_idx)

  model, epoch_losses = _train_nn(X_train_resampled, y_train_resampled, input_dim)
  y_pred, y_prob = _predict(model, XtestScaled)
  metrics = compute_metrics(y_testr, y_pred, y_prob)
  metrics["split"] = split_num
  nn_results.append(metrics)
  split_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
  all_epoch_losses[split_num] = epoch_losses

nn_results = pd.DataFrame(nn_results)

fig, ax = plt.subplots(figsize=(10,5))
color_splits = ["#056517", "#de1a24", "#3b75e9", "#cad5ed"]
for split_num in sorted(all_epoch_losses.keys()):
  losses = all_epoch_losses[split_num]
  color = color_splits[(split_num-1)%len(color_splits)]
  ax.plot(range(1, len(losses) +1), losses, label=f"Split {split_num}", color=color)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Over Epochs")
ax.legend()
save_fig(fig, "Training_Loss.png")

plot_metrics_bar(nn_results,splits, "FFNN", "FFNN_Metrics")
plot_roc_curve(splits, Xfeatures, y, make_predictor("y_prob"), "FFNN", "FFNN_ROC_Curve")
confusion_matrices(splits, Xfeatures, y, make_predictor("y_pred"), "FFNN", "FFNN_Confusion_Matrix")